## **Voting Ensemble Classifier**

### **Topic Roadmap**

**1. Prepare a classification dataset**

**2. Train base classifiers**

**3. Compare hard and soft voting**

**4. Evaluate the ensemble**

**5. Key revision notes**

## **1. Dataset and Split**

Voting combines predictions from several different classifiers. A stratified split gives a fair test of the ensemble.

In [6]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

import warnings
warnings.filterwarnings("ignore")

from sklearn.datasets import load_breast_cancer

In [2]:
data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

## **2. Define Base Classifiers**

The logistic-regression pipeline scales features, while the tree and random forest do not require scaling. Diversity among models is useful for voting.

In [3]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

logistic = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
tree = DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE)
forest = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)

In [4]:
for name, model in [("Logistic regression", logistic), ("Decision tree", tree), ("Random forest", forest)]:
    model.fit(X_train, y_train)
    print(name, f"test accuracy = {model.score(X_test, y_test):.3f}")

Logistic regression test accuracy = 0.982
Decision tree test accuracy = 0.921
Random forest test accuracy = 0.956


## **3. Hard and Soft Voting**

Hard voting selects the majority class. Soft voting averages class probabilities and requires base estimators that implement `predict_proba`.

In [5]:
from sklearn.ensemble import VotingClassifier

hard_voter = VotingClassifier(
    estimators=[("lr", logistic), ("tree", tree), ("rf", forest)], voting="hard"
)
soft_voter = VotingClassifier(
    estimators=[("lr", logistic), ("tree", tree), ("rf", forest)], voting="soft"
)
for name, model in [("Hard voting", hard_voter), ("Soft voting", soft_voter)]:
    model.fit(X_train, y_train)
    print(name, f"test accuracy = {model.score(X_test, y_test):.3f}")

Hard voting test accuracy = 0.974
Soft voting test accuracy = 0.947


### **Key Revision Notes**

- Hard voting uses predicted class labels.
- Soft voting averages predicted probabilities and can perform better when probabilities are informative.
- Base estimators should be reasonably diverse and individually useful.
- Scale only the models that need it; a pipeline keeps preprocessing inside cross-validation.